In [3]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017")
print(client.list_database_names())

db = client['worldCup']
print(db.list_collection_names())        

coll = db['matches']
doc = coll.find_one()
print(doc)                               

['admin', 'config', 'local', 'worldCup']
['matches']
{'_id': ObjectId('69bc334b571647f19d2ab786'), 'tournament': '1930--uruguay', 'stage': 'Group 1', 'date': datetime.datetime(1930, 7, 13, 0, 0), 'home_team': 'France', 'away_team': 'Mexico', 'home_score': 4, 'away_score': 1, 'venue': 'Estadio Pocitos, Montevideo', 'winner': 'France', 'home_elo': 1500.0, 'away_elo': 1500.0, 'home_elo_after': 1530.0, 'away_elo_after': 1470.0, 'elo_diff': 0.0, 'home_rank': None, 'away_rank': None, 'home_rank_obj': None, 'away_rank_obj': None, 'rank_source': 'FIFA', 'home_days_since_last_match': None, 'away_days_since_last_match': None, 'rank_diff': None, 'is_home_favourite': None, 'ingested_at': datetime.datetime(2026, 3, 19, 17, 32, 59, 957000)}


In [12]:
datos = list(coll.find(
    {},
    {
    "_id": 0,
    "home_team": 1,
    "away_team": 1,
    "tournament": 1,
    "stage": 1,
    "date": 1,
    "home_score": 1,
    "away_score": 1,
    "venue": 1,
    "elo_diff": 1,
    "winner": 1,
    "home_days_since_last_match": 1,
    "away_days_since_last_match": 1
    }
))
print(datos)

[{'tournament': '1930--uruguay', 'stage': 'Group 1', 'date': datetime.datetime(1930, 7, 13, 0, 0), 'home_team': 'France', 'away_team': 'Mexico', 'home_score': 4, 'away_score': 1, 'venue': 'Estadio Pocitos, Montevideo', 'winner': 'France', 'elo_diff': 0.0, 'home_days_since_last_match': None, 'away_days_since_last_match': None}, {'tournament': '1930--uruguay', 'stage': 'Group 4', 'date': datetime.datetime(1930, 7, 13, 0, 0), 'home_team': 'United States', 'away_team': 'Belgium', 'home_score': 3, 'away_score': 0, 'venue': 'Estadio Parque Central, Montevideo', 'winner': 'United States', 'elo_diff': 0.0, 'home_days_since_last_match': None, 'away_days_since_last_match': None}, {'tournament': '1930--uruguay', 'stage': 'Group 2', 'date': datetime.datetime(1930, 7, 14, 0, 0), 'home_team': 'Yugoslavia', 'away_team': 'Brazil', 'home_score': 2, 'away_score': 1, 'venue': 'Estadio Parque Central, Montevideo', 'winner': 'Yugoslavia', 'elo_diff': 0.0, 'home_days_since_last_match': None, 'away_days_sinc

Hay que armar cada unas de las columnas y tambien extraer sus datos

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from pymongo import MongoClient
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import numpy as np


client = MongoClient("mongodb://localhost:27017/")
db = client["worldCup"]
coll = db["matches"]

datos = list(coll.find({}, {
    "_id": 0,
    "home_team": 1,
    "away_team": 1,
    "tournament": 1,
    "stage": 1,
    "home_score": 1,
    "away_score": 1,
    "elo_diff": 1,
    "home_days_since_last_match": 1,
    "away_days_since_last_match": 1,
    "winner": 1
}))


le_team = LabelEncoder()
le_tournament = LabelEncoder()
le_stage = LabelEncoder()
le_winner = LabelEncoder()

home_team = le_team.fit_transform([d["home_team"] for d in datos])
away_team = le_team.fit_transform([d["away_team"] for d in datos])
tournament = le_tournament.fit_transform([d["tournament"] for d in datos])
stage = le_stage.fit_transform([d["stage"] for d in datos])
y = le_winner.fit_transform([d["winner"] for d in datos])

X = np.column_stack([
    home_team,
    away_team,
    tournament,
    stage,
    [d["home_score"] for d in datos],
    [d["away_score"] for d in datos],
    [d["elo_diff"] for d in datos],
    [d["home_days_since_last_match"] if d["home_days_since_last_match"] is not None else 0 for d in datos],
    [d["away_days_since_last_match"] if d["away_days_since_last_match"] is not None else 0 for d in datos]
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

model = RandomForestClassifier()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)


print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.5578947368421052


Predicción real

In [26]:
home = le_team.transform(["France"])[0]
away = le_team.transform(["Spain"])[0]
import numpy as np

import numpy as np
home = le_team.transform(["France"])[0]
away = le_team.transform(["Spain"])[0]
tournament = le_tournament.transform(["1930--uruguay"])[0]
stage = le_stage.transform(["Final"])[0]
X_new = np.array([[
    home,
    away,
    tournament,
    stage,
    0,   # home_score (si no lo sabes)
    0,   # away_score
    50,  # elo_diff
    5,   # home_days_since_last_match
    6    # away_days_since_last_match
]])

X_new = np.array([[home, away, 50, 1]]) 
model.predict(X_new) 
winner = le_winner.inverse_transform(model.predict(X_new))
print(winner[0])

ValueError: X has 4 features, but RandomForestClassifier is expecting 9 features as input.